# Setup

In [ ]:
import os
from tqdm.auto import tqdm
from joblib import Parallel, delayed
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'axes.titleweight': 'bold',
})
C_MEN = '#2c7bb6'
C_WOMEN = '#d7191c'
C_GRAY = '#636363'

INPUT_ROOT = Path('/kaggle/input')
DATA_DIR = None
for p in INPUT_ROOT.rglob('MTeams.csv'):
    DATA_DIR = p.parent
    break
if DATA_DIR is None:
    raise FileNotFoundError(f'MTeams.csv not found under {INPUT_ROOT}')

# Data Loading

In [ ]:
m_teams = pd.read_csv(DATA_DIR / 'MTeams.csv')
m_seeds = pd.read_csv(DATA_DIR / 'MNCAATourneySeeds.csv')
m_tourney = pd.read_csv(DATA_DIR / 'MNCAATourneyCompactResults.csv')
m_reg = pd.read_csv(DATA_DIR / 'MRegularSeasonCompactResults.csv')
massey = pd.read_csv(DATA_DIR / 'MMasseyOrdinals.csv')
w_teams = pd.read_csv(DATA_DIR / 'WTeams.csv')
w_seeds = pd.read_csv(DATA_DIR / 'WNCAATourneySeeds.csv')
w_tourney = pd.read_csv(DATA_DIR / 'WNCAATourneyCompactResults.csv')
w_reg = pd.read_csv(DATA_DIR / 'WRegularSeasonCompactResults.csv')
sub_stage1 = pd.read_csv(DATA_DIR / 'SampleSubmissionStage1.csv')

def parse_seed(s):
    return int(''.join(filter(str.isdigit, s)))

m_seeds['SeedNum'] = m_seeds['Seed'].apply(parse_seed)
w_seeds['SeedNum'] = w_seeds['Seed'].apply(parse_seed)

# Feature Engineering

In [ ]:
def build_season_stats(reg_df):
    w = reg_df[['Season','WTeamID','WScore','LScore']].copy()
    w.columns = ['Season','TeamID','ScoreFor','ScoreAgainst']
    w['Win'] = 1
    l = reg_df[['Season','LTeamID','LScore','WScore']].copy()
    l.columns = ['Season','TeamID','ScoreFor','ScoreAgainst']
    l['Win'] = 0
    df = pd.concat([w, l], ignore_index=True)
    df['Margin'] = df['ScoreFor'] - df['ScoreAgainst']
    stats = df.groupby(['Season','TeamID']).agg(
        Games=('Win','count'),
        Wins=('Win','sum'),
        AvgScore=('ScoreFor','mean'),
        AvgMargin=('Margin','mean'),
    ).reset_index()
    stats['WinRate'] = stats['Wins'] / stats['Games']
    return stats

m_stats = build_season_stats(m_reg)
w_stats = build_season_stats(w_reg)

massey_agg = (
    massey[massey['SystemName'] == 'MOR']
    .sort_values(['Season','RankingDayNum'])
    .groupby(['Season','TeamID']).last()
    .reset_index()
    [['Season','TeamID','OrdinalRank']].rename(columns={'OrdinalRank':'MOR'})
)

def get_feats(season, team_id, stats, seed_df, mor_df=None):
    s_row = stats[(stats.Season == season) & (stats.TeamID == team_id)]
    e_row = seed_df[(seed_df.Season == season) & (seed_df.TeamID == team_id)]
    win_rate = s_row['WinRate'].values[0] if len(s_row) else np.nan
    avg_margin = s_row['AvgMargin'].values[0] if len(s_row) else np.nan
    avg_score = s_row['AvgScore'].values[0] if len(s_row) else np.nan
    seed_num = e_row['SeedNum'].values[0] if len(e_row) else np.nan
    mor = np.nan
    if mor_df is not None:
        m_row = mor_df[(mor_df.Season == season) & (mor_df.TeamID == team_id)]
        mor = m_row['MOR'].values[0] if len(m_row) else np.nan
    return win_rate, avg_margin, avg_score, seed_num, mor

def build_train_df(tourney, stats, seed_df, mor_df=None):
    rows = []
    for _, r in tourney.iterrows():
        s = r['Season']
        t1, t2 = sorted([r['WTeamID'], r['LTeamID']])
        label = 1 if r['WTeamID'] == t1 else 0
        f1 = get_feats(s, t1, stats, seed_df, mor_df)
        f2 = get_feats(s, t2, stats, seed_df, mor_df)
        rows.append(dict(
            Season=s, T1=t1, T2=t2,
            T1_WinRate=f1[0], T2_WinRate=f2[0],
            T1_Margin=f1[1], T2_Margin=f2[1],
            T1_Score=f1[2], T2_Score=f2[2],
            T1_Seed=f1[3], T2_Seed=f2[3],
            T1_MOR=f1[4], T2_MOR=f2[4],
            Label=label
        ))
    df = pd.DataFrame(rows)
    df['SeedDiff'] = df['T1_Seed'] - df['T2_Seed']
    df['WinRateDiff'] = df['T1_WinRate'] - df['T2_WinRate']
    df['MarginDiff'] = df['T1_Margin'] - df['T2_Margin']
    df['ScoreDiff'] = df['T1_Score'] - df['T2_Score']
    df['MORDiff'] = df['T1_MOR'] - df['T2_MOR']
    
    for c in ['T1_Seed', 'T2_Seed']:
        df[c] = df[c].fillna(-1).astype(int).astype('category')
    return df

m_train = build_train_df(m_tourney, m_stats, m_seeds, massey_agg)
w_train = build_train_df(w_tourney, w_stats, w_seeds, mor_df=None)

CAT_COLS = ['T1_Seed', 'T2_Seed']
NUM_COLS = ['SeedDiff','WinRateDiff','MarginDiff','ScoreDiff','MORDiff',
            'T1_WinRate','T2_WinRate','T1_Margin','T2_Margin','T1_MOR','T2_MOR']
FEAT_COLS = NUM_COLS + CAT_COLS

# Modeling (LGBM + XGB + CatBoost Ensemble)

In [ ]:
LGB_PARAMS = dict(
    n_estimators=300, 
    learning_rate=0.05, 
    num_leaves=31,
    min_child_samples=20, 
    subsample=0.8, 
    colsample_bytree=0.8,
    random_state=42, 
    n_jobs=-1
)

XGB_PARAMS = dict(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    enable_categorical=True,
    tree_method='hist',
    n_jobs=-1
)

CAT_PARAMS = dict(
    iterations=300,
    learning_rate=0.05,
    depth=5,
    random_state=42,
    verbose=0,
    thread_count=-1,
    task_type='CPU'
)

def run_ensemble_cv(train_df, feat_cols, cat_cols, n_folds=5):
    df = train_df.dropna(subset=['SeedDiff','Label']).copy()
    num_c = [c for c in feat_cols if c not in cat_cols]
    global_med = df[num_c].median().fillna(0)
    df[num_c] = df[num_c].fillna(global_med)
    X, y = df[feat_cols], df['Label'].values

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_scores, oof_preds, oof_labels = [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        lgb_m = lgb.LGBMClassifier(**LGB_PARAMS)
        lgb_m.fit(X_tr, y_tr, categorical_feature=cat_cols)
        prob_lgb = lgb_m.predict_proba(X_va)[:, 1]
        
        xgb_m = xgb.XGBClassifier(**XGB_PARAMS)
        xgb_m.fit(X_tr, y_tr)
        prob_xgb = xgb_m.predict_proba(X_va)[:, 1]
        
        cat_m = CatBoostClassifier(**CAT_PARAMS, cat_features=cat_cols)
        cat_m.fit(X_tr, y_tr)
        prob_cat = cat_m.predict_proba(X_va)[:, 1]

        prob = (prob_lgb + prob_xgb + prob_cat) / 3
        
        ll = log_loss(y_va, prob)
        fold_scores.append(ll)
        oof_preds.extend(prob.tolist())
        oof_labels.extend(y_va.tolist())

    return np.mean(fold_scores), np.std(fold_scores), oof_preds, oof_labels

fc_m = [c for c in FEAT_COLS if c in m_train.columns]
cc_m = [c for c in CAT_COLS if c in m_train.columns]
mean_m, std_m, oof_p_m, oof_l_m = run_ensemble_cv(m_train, fc_m, cc_m)

fc_w = [c for c in FEAT_COLS if c in w_train.columns]
cc_w = [c for c in CAT_COLS if c in w_train.columns]
mean_w, std_w, oof_p_w, oof_l_w = run_ensemble_cv(w_train, fc_w, cc_w)

# Evaluation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(oof_p_m, bins=40, color=C_MEN, alpha=0.6, edgecolor='white', density=True, label="Men's")
ax.hist(oof_p_w, bins=40, color=C_WOMEN, alpha=0.6, edgecolor='white', density=True, label="Women's")
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.2)
ax.set_title('OOF Prediction Distribution')
ax.set_xlabel('Predicted probability P(T1 wins)')
ax.set_ylabel('Density')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Deployment and Submission

In [ ]:
def predict_test(train_df, test_df, feat_cols, cat_cols):
    df_train = train_df.dropna(subset=['SeedDiff','Label']).copy()
    df_test = test_df.copy()
    
    num_c = [c for c in feat_cols if c not in cat_cols]
    global_med = df_train[num_c].median().fillna(0)
    df_train[num_c] = df_train[num_c].fillna(global_med)
    df_test[num_c] = df_test[num_c].fillna(global_med)
    
    X_tr, y_tr = df_train[feat_cols], df_train['Label'].values
    X_te = df_test[feat_cols]
    
    lgb_m = lgb.LGBMClassifier(**LGB_PARAMS)
    lgb_m.fit(X_tr, y_tr, categorical_feature=cat_cols)
    prob_lgb = lgb_m.predict_proba(X_te)[:, 1]
    
    xgb_m = xgb.XGBClassifier(**XGB_PARAMS)
    xgb_m.fit(X_tr, y_tr)
    prob_xgb = xgb_m.predict_proba(X_te)[:, 1]
    
    cat_m = CatBoostClassifier(**CAT_PARAMS, cat_features=cat_cols)
    cat_m.fit(X_tr, y_tr)
    prob_cat = cat_m.predict_proba(X_te)[:, 1]
    
    return (prob_lgb + prob_xgb + prob_cat) / 3

sub_stage1['Season'] = sub_stage1['ID'].apply(lambda x: int(x.split('_')[0]))
sub_stage1['T1'] = sub_stage1['ID'].apply(lambda x: int(x.split('_')[1]))
sub_stage1['T2'] = sub_stage1['ID'].apply(lambda x: int(x.split('_')[2]))

m_sub = sub_stage1[sub_stage1['T1'].isin(m_teams['TeamID'])].copy()
w_sub = sub_stage1[sub_stage1['T1'].isin(w_teams['TeamID'])].copy()

def process_row(s, t1, t2, stats, seed_df, mor_df):
    f1 = get_feats(s, t1, stats, seed_df, mor_df)
    f2 = get_feats(s, t2, stats, seed_df, mor_df)
    return dict(
        T1_WinRate=f1[0], T2_WinRate=f2[0],
        T1_Margin=f1[1], T2_Margin=f2[1],
        T1_Score=f1[2], T2_Score=f2[2],
        T1_Seed=f1[3], T2_Seed=f2[3],
        T1_MOR=f1[4], T2_MOR=f2[4]
    )

def extract_sub_feats(sub_df, stats, seed_df, mor_df=None):
    rows = Parallel(n_jobs=-1, backend="threading")(
        delayed(process_row)(r['Season'], r['T1'], r['T2'], stats, seed_df, mor_df)
        for _, r in tqdm(sub_df.iterrows(), total=len(sub_df), desc="Extracting Features")
    )
    
    df = pd.DataFrame(rows, index=sub_df.index)
    for c in df.columns:
        sub_df[c] = df[c]
    
    sub_df['SeedDiff'] = sub_df['T1_Seed'] - sub_df['T2_Seed']
    sub_df['WinRateDiff'] = sub_df['T1_WinRate'] - sub_df['T2_WinRate']
    sub_df['MarginDiff'] = sub_df['T1_Margin'] - sub_df['T2_Margin']
    sub_df['ScoreDiff'] = sub_df['T1_Score'] - sub_df['T2_Score']
    sub_df['MORDiff'] = sub_df['T1_MOR'] - sub_df['T2_MOR']
    for c in ['T1_Seed', 'T2_Seed']:
        sub_df[c] = sub_df[c].fillna(-1).astype(int).astype('category')
    return sub_df

print("Processing Men's submission features...")
m_sub = extract_sub_feats(m_sub, m_stats, m_seeds, massey_agg)

print("Processing Women's submission features...")
w_sub = extract_sub_feats(w_sub, w_stats, w_seeds, None)

print("Running inferences...")
m_sub['Pred'] = predict_test(m_train, m_sub, fc_m, cc_m)
w_sub['Pred'] = predict_test(w_train, w_sub, fc_w, cc_w)

sub_final = pd.concat([m_sub, w_sub]).sort_index()
sub_final['Pred'] = sub_final['Pred'].clip(0.025, 0.975)
submission = sub_final[['ID', 'Pred']]
submission.to_csv('submission.csv', index=False)
print("Submission saved successfully!")